In [1]:
# CELL 1 - imports + paths + constants
# HOW TO CHECK: runs silently. Last line prints the device.
import os, sys, json, time, copy, pickle, shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

BASE = r'D:\FYP\Eco-Twin'
V2 = os.path.join(BASE, 'v2_clean')
OUT = os.path.join(V2, 'ecotwin_outputs')
SCALER_DIR = os.path.join(OUT, 'scalers')
MODEL_DIR = os.path.join(OUT, 'models')
RESULTS_DIR = os.path.join(OUT, 'results')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

INPUT_WINDOW, HORIZON, BATCH_SIZE, LOSS_VOL_ALPHA = 48, 24, 16, 1.0
HIDDEN_DIM, GRU_LAYERS, ATTN_HEADS, DROPOUT, GCN_HOPS = 32, 1, 4, 0.2, 2
ALL_SEEDS = [42, 7, 123, 2024, 999, 2025, 77]
BASE_FEATS = ['pm2.5', 'pm10', 'no2', 'so2', 'co', 'o3', 'rh', 'at']
REST = ['pm_roll6_mean', 'pm_roll6_std', 'pm_roll24_mean', 'pm_roll24_std', 'pm_delta3',
        'city_mean_pm', 'city_max_pm', 'pm_minus_city',
        'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'mon_sin', 'mon_cos', 'is_weekend']
TIME_COLS = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'mon_sin', 'mon_cos', 'is_weekend']
print('device:', DEVICE)


device: cuda


In [2]:
# CELL 2 - model (same architecture as nb14/nb15)
# HOW TO CHECK: runs silently.
class Net(nn.Module):
    def __init__(self, in_features, hidden_dim, gru_layers, attn_heads, forecast_horizon,
                 n_stations, dropout=0.2, n_fut=10, gcn_hops=2, graph_mode='dynamic', static_adj=None):
        super().__init__()
        H = hidden_dim
        self.input_proj = nn.Linear(in_features, H)
        self.gcn_lin = nn.ModuleList(nn.Linear(H, H) for _ in range(gcn_hops))
        self.gcn_norm = nn.ModuleList(nn.LayerNorm(H) for _ in range(gcn_hops))
        self.act = nn.ReLU()
        self.gru = nn.GRU(H, H, gru_layers, batch_first=True, dropout=dropout if gru_layers > 1 else 0.0)
        self.attn = nn.MultiheadAttention(H, attn_heads, dropout=dropout, batch_first=True)
        self.attn_norm = nn.LayerNorm(H)
        self.dropout = nn.Dropout(dropout)
        self.fut_enc = nn.Sequential(nn.Linear(n_fut, H), nn.ReLU(), nn.Linear(H, H))
        self.decoder = nn.Sequential(nn.Linear(H * 3, H), nn.ReLU(), nn.Dropout(dropout), nn.Linear(H, 1))
        self.anchor_gate = nn.Parameter(torch.linspace(1.0, 0.3, forecast_horizon))
        self.H, self.FH, self.graph_mode = H, forecast_horizon, graph_mode
        sa = torch.eye(n_stations) if static_adj is None else torch.from_numpy(static_adj.astype('float32'))
        self.register_buffer('static_adj', sa, persistent=False)

    @staticmethod
    def _n(A):
        return A / A.sum(-1, keepdim=True).clamp(min=1e-6)

    def forward(self, X, At, Afc, Fut, Anchor):
        B, Tin, n, _ = X.shape
        dev = X.device
        I = torch.eye(n, device=dev)
        h = self.act(self.input_proj(X))
        if self.graph_mode == 'none':
            for lin, nrm in zip(self.gcn_lin, self.gcn_norm):
                h = self.act(nrm(lin(h))) + h
        else:
            if self.graph_mode == 'dynamic':
                A = self._n(At + I.view(1, 1, n, n))
            else:
                A = self._n((self.static_adj + I).view(1, 1, n, n)).expand(B, Tin, n, n)
            for lin, nrm in zip(self.gcn_lin, self.gcn_norm):
                h = self.act(nrm(lin(torch.matmul(A, h)))) + h
        hf = h.permute(0, 2, 1, 3).reshape(B * n, Tin, self.H)
        gg, _ = self.gru(hf)
        gg = self.dropout(gg)
        aa, _ = self.attn(gg, gg, gg)
        ctx = self.attn_norm(aa + gg)[:, -1, :].reshape(B, n, self.H)
        ctx_rep = ctx.unsqueeze(1).expand(B, self.FH, n, self.H)
        if self.graph_mode == 'none':
            ctx_fc = ctx_rep
        elif self.graph_mode == 'dynamic':
            ctx_fc = torch.matmul(self._n(Afc + I.view(1, 1, n, n)), ctx_rep)
        else:
            ctx_fc = torch.matmul(self._n((self.static_adj + I).view(1, 1, n, n)), ctx_rep)
        fut_emb = self.fut_enc(Fut)
        resid = self.decoder(torch.cat([ctx_rep, ctx_fc, fut_emb], dim=-1)).squeeze(-1)
        return resid + Anchor.unsqueeze(1) * self.anchor_gate.view(1, self.FH, 1)


In [3]:
# CELL 3 - dataset class + data pipeline builder
# HOW TO CHECK: runs silently.
class DS(Dataset):
    def __init__(self, ctx, starts, weights=None):
        self.c = ctx
        self.s = np.asarray(starts, np.int64)
        self.w = (weights if weights is not None else np.ones(len(self.s), np.float32)).astype(np.float32)

    def __len__(self):
        return len(self.s)

    def __getitem__(self, k):
        c = self.c
        g = int(self.s[k]); i1 = g + INPUT_WINDOW; i2 = i1 + HORIZON
        return (torch.from_numpy(c['Xs'][g:i1].copy()),
                torch.from_numpy(c['adj'][g:i1].copy()),
                torch.from_numpy(c['adj'][i1:i2].copy()),
                torch.from_numpy(np.ascontiguousarray(c['Xs'][i1:i2][:, :, c['FUT_IDX']])),
                torch.from_numpy(c['Yl'][i1 - 1].copy()),
                torch.from_numpy(c['Yl'][i1:i2].copy()),
                torch.tensor(self.w[k], dtype=torch.float32))


def build(wind_in_features, adj_file):
    df = pd.read_csv(os.path.join(V2, 'EcoTwin_Final_Dataset.csv'), parse_dates=['Timestamp'])
    df['Station'] = df['Station'].str.strip()
    df = df.sort_values(['Station', 'Timestamp']).reset_index(drop=True)
    df_st = pd.read_excel(os.path.join(V2, 'Delhi_list_v2.xlsx'))
    df_st.columns = df_st.columns.str.strip().str.lower()
    df_st['station'] = df_st['station'].str.strip()
    names = df_st['station'].tolist()
    N = len(names)
    T = len(np.array(sorted(df['Timestamp'].unique())))
    era5 = np.load(os.path.join(V2, 'openmeteo_reanalysis_uv.npy')).astype(np.float32)
    era5_spd = np.hypot(era5[..., 0], era5[..., 1])
    adj = np.load(os.path.join(V2, adj_file)).astype(np.float32)
    static_adj = np.load(os.path.join(V2, 'static_adj_matrix.npy')).astype(np.float32)
    np.fill_diagonal(static_adj, 0.0)
    assert adj.shape == (T, N, N), adj.shape

    g = df.groupby('Station', sort=False)
    df['pm_roll6_mean'] = g['pm2.5'].transform(lambda s: s.rolling(6, min_periods=1).mean())
    df['pm_roll6_std'] = g['pm2.5'].transform(lambda s: s.rolling(6, min_periods=1).std()).fillna(0.0)
    df['pm_roll24_mean'] = g['pm2.5'].transform(lambda s: s.rolling(24, min_periods=1).mean())
    df['pm_roll24_std'] = g['pm2.5'].transform(lambda s: s.rolling(24, min_periods=1).std()).fillna(0.0)
    df['pm_delta3'] = g['pm2.5'].transform(lambda s: s - s.shift(3)).fillna(0.0)
    tsg = df.groupby('Timestamp', sort=False)['pm2.5']
    df['city_mean_pm'] = tsg.transform('mean')
    df['city_max_pm'] = tsg.transform('max')
    df['pm_minus_city'] = df['pm2.5'] - df['city_mean_pm']
    hh = df['Timestamp'].dt.hour; dow = df['Timestamp'].dt.dayofweek; mon = df['Timestamp'].dt.month
    df['hour_sin'] = np.sin(2 * np.pi * hh / 24); df['hour_cos'] = np.cos(2 * np.pi * hh / 24)
    df['dow_sin'] = np.sin(2 * np.pi * dow / 7); df['dow_cos'] = np.cos(2 * np.pi * dow / 7)
    df['mon_sin'] = np.sin(2 * np.pi * (mon - 1) / 12); df['mon_cos'] = np.cos(2 * np.pi * (mon - 1) / 12)
    df['is_weekend'] = (dow >= 5).astype(float)

    Xr = np.zeros((T, N, len(BASE_FEATS + REST)), dtype=np.float32)
    for fi, col in enumerate(BASE_FEATS + REST):
        wide = df.pivot_table(index='Timestamp', columns='Station', values=col).reindex(columns=names)
        Xr[:, :, fi] = np.nan_to_num(wide.values.astype(np.float32))

    if wind_in_features:
        FEATURE_COLS = BASE_FEATS + ['u_wind', 'v_wind', 'wspd'] + REST
        X_raw = np.concatenate([Xr[:, :, :8], era5[:, :, 0:1], era5[:, :, 1:2], era5_spd[:, :, None], Xr[:, :, 8:]],
                               axis=2).astype(np.float32)
    else:
        FEATURE_COLS = BASE_FEATS + REST
        X_raw = Xr.astype(np.float32)
    F = len(FEATURE_COLS)
    Y_raw = X_raw[:, :, 0].copy()
    TIME_IDX = [FEATURE_COLS.index(c) for c in TIME_COLS]
    FUT_IDX = TIME_IDX + ([FEATURE_COLS.index(c) for c in ['u_wind', 'v_wind', 'wspd']] if wind_in_features else [])

    train_end, val_end = int(T * 0.70), int(T * 0.85)

    def starts(lo, hi):
        return np.arange(lo, lo + (hi - lo) - (INPUT_WINDOW + HORIZON) + 1)
    tr_s, va_s, te_s = starts(0, train_end), starts(train_end, val_end), starts(val_end, T)

    tr_mask = np.zeros(T, bool); tr_mask[:train_end] = True
    feat_scaler = StandardScaler().fit(X_raw[tr_mask].reshape(-1, F))
    Xs = feat_scaler.transform(X_raw.reshape(-1, F)).reshape(T, N, F).astype(np.float32)
    log_scaler = StandardScaler().fit(np.log1p(np.clip(Y_raw[tr_mask].reshape(-1, 1), 0, None)))

    def to_log(a):
        return log_scaler.transform(np.log1p(np.clip(a, 0, None)).reshape(-1, 1)).reshape(a.shape).astype(np.float32)

    def from_log(a):
        return np.expm1(log_scaler.inverse_transform(a.reshape(-1, 1)).reshape(a.shape))

    Yl = to_log(Y_raw)
    swing = np.array([(Y_raw[s + INPUT_WINDOW:s + INPUT_WINDOW + HORIZON].max(0) -
                       Y_raw[s + INPUT_WINDOW:s + INPUT_WINDOW + HORIZON].min(0)).mean() for s in tr_s], np.float32)
    med = float(np.median(swing))
    tr_w = np.clip(1.0 + LOSS_VOL_ALPHA * (swing / (med + 1e-6)), 0.5, 4.0).astype(np.float32)
    tr_w *= len(tr_w) / tr_w.sum()

    ctx = dict(Xs=Xs, Yl=Yl, adj=adj, static_adj=static_adj, FUT_IDX=FUT_IDX, F=F, n_fut=len(FUT_IDX), N=N, T=T,
               FEATURE_COLS=FEATURE_COLS, feat_scaler=feat_scaler, log_scaler=log_scaler,
               to_log=to_log, from_log=from_log, era5_spd=era5_spd, te_starts=te_s)
    ctx['tr'] = DataLoader(DS(ctx, tr_s, tr_w), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    ctx['va'] = DataLoader(DS(ctx, va_s), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    ctx['te'] = DataLoader(DS(ctx, te_s), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    print(f"[build] wind_in_features={wind_in_features} F={F} n_fut={ctx['n_fut']} adj={adj_file} "
          f"windows train/val/test {len(tr_s)}/{len(va_s)}/{len(te_s)}", flush=True)
    return ctx


In [4]:
# CELL 4 - train / predict / metric helpers
# HOW TO CHECK: runs silently.
def make_model(ctx, mode):
    return Net(ctx['F'], HIDDEN_DIM, GRU_LAYERS, ATTN_HEADS, HORIZON, ctx['N'], DROPOUT,
               ctx['n_fut'], GCN_HOPS, graph_mode=mode, static_adj=ctx['static_adj']).to(DEVICE)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    P, A = [], []
    for X, At, Afc, Fut, anc, y, w in loader:
        P.append(model(X.to(DEVICE), At.to(DEVICE), Afc.to(DEVICE), Fut.to(DEVICE), anc.to(DEVICE)).cpu().numpy())
        A.append(y.numpy())
    return np.concatenate(P), np.concatenate(A)


def metrics(ctx, pl, al):
    pr, ar = ctx['from_log'](pl), ctx['from_log'](al)
    return dict(rmse=float(np.sqrt(mean_squared_error(ar.ravel(), pr.ravel()))),
                mae=float(mean_absolute_error(ar.ravel(), pr.ravel())),
                r2=float(r2_score(ar.ravel(), pr.ravel())))


def weighted_huber(pred, target, w, delta=1.0):
    absr = (pred - target).abs()
    quad = torch.clamp(absr, max=delta)
    per = (0.5 * quad ** 2 + delta * (absr - quad)).mean(dim=(1, 2))
    return (per * w).mean() / w.mean().clamp(min=1e-6)


def run_epoch(model, loader, opt=None, sched=None):
    tr = opt is not None
    model.train(tr)
    tot = n = 0
    for X, At, Afc, Fut, anc, y, w in loader:
        X, At, Afc = X.to(DEVICE), At.to(DEVICE), Afc.to(DEVICE)
        Fut, anc, y, w = Fut.to(DEVICE), anc.to(DEVICE), y.to(DEVICE), w.to(DEVICE)
        with torch.set_grad_enabled(tr):
            out = model(X, At, Afc, Fut, anc)
            loss = weighted_huber(out, y, w if tr else torch.ones_like(w))
        if tr:
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            if sched:
                sched.step()
        tot += loss.item() * X.size(0); n += X.size(0)
    return tot / n


def train_one(ctx, mode, seed, max_epochs=60, patience=12, lr=1e-3):
    torch.manual_seed(seed); np.random.seed(seed)
    model = make_model(ctx, mode)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    steps = len(ctx['tr']) * max_epochs
    warm = max(1, int(0.05 * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: s / warm if s < warm else 0.5 * (1 + np.cos(np.pi * (s - warm) / max(1, steps - warm))))
    best, best_state, ctr = 1e9, None, 0
    for ep in range(1, max_epochs + 1):
        t0 = time.time()
        trl = run_epoch(model, ctx['tr'], opt, sched)
        vl = run_epoch(model, ctx['va'])
        if vl < best - 1e-5:
            best, best_state, ctr = vl, copy.deepcopy(model.state_dict()), 0
        else:
            ctr += 1
        if ep % 4 == 0 or ctr >= patience:
            print(f'    ep {ep:>2} train {trl:.4f} val {vl:.4f} ({time.time() - t0:.0f}s) best {best:.4f}', flush=True)
        if ctr >= patience:
            break
    model.load_state_dict(best_state)
    return model


def load_ckpts(ctx, mode, prefix, seeds):
    preds, actual = [], None
    for sd in seeds:
        m = make_model(ctx, mode)
        m.load_state_dict(torch.load(os.path.join(MODEL_DIR, f'{prefix}_seed{sd}.pt'), map_location=DEVICE))
        p, a = predict(m, ctx['te'])
        preds.append(p); actual = a
        del m
    return preds, actual


In [5]:
# CELL 5 - TRAIN the controls (this is the long one: ~45-60 min, keep this tab open, no other GPU jobs)
# Same 23 inputs as v3b (NO wind inputs, calendar-only decoder). Only the graph differs.
# HOW TO CHECK: first it prints  verify v3b 7-seed test RMSE 62.21 (expected 62.21)  - if not, STOP.
#               then 5 seeds of no-graph, then 5 seeds of static-graph, each printing "test RMSE".
ctx = build(False, 'dynamic_adj_watgcn_v3b.npy')
seeds5 = ALL_SEEDS[:5]

dp, actual = load_ckpts(ctx, 'dynamic', 'watgcn_v3b', ALL_SEEDS)
print(f'verify v3b 7-seed test RMSE {metrics(ctx, np.mean(dp, axis=0), actual)["rmse"]:.2f} (expected 62.21)', flush=True)

preds = {'v3b dynamic (35km advection)': dp[:5]}
for mode, label in [('none', 'no-graph (no wind)'), ('static', 'static-GCN (no wind)')]:
    ps = []
    for sd in seeds5:
        print(f'\n[{label}] seed {sd}', flush=True)
        m = train_one(ctx, mode, sd)
        p, a = predict(m, ctx['te']); ps.append(p)
        print(f'  -> seed {sd} test RMSE {metrics(ctx, p, a)["rmse"]:.2f}', flush=True)
        del m; torch.cuda.empty_cache()
    preds[label] = ps
print('\nTRAINING DONE - now run the next cell')


[build] wind_in_features=False F=23 n_fut=7 adj=dynamic_adj_watgcn_v3b.npy windows train/val/test 13873/2917/2918
verify v3b 7-seed test RMSE 62.21 (expected 62.21)

[no-graph (no wind)] seed 42
    ep  4 train 0.1353 val 0.1335 (19s) best 0.1335
    ep  8 train 0.1207 val 0.1359 (17s) best 0.1335
    ep 12 train 0.1091 val 0.1486 (17s) best 0.1335
    ep 16 train 0.1011 val 0.1514 (18s) best 0.1335
  -> seed 42 test RMSE 63.33

[no-graph (no wind)] seed 7
    ep  4 train 0.1351 val 0.1369 (18s) best 0.1363
    ep  8 train 0.1221 val 0.1382 (17s) best 0.1363
    ep 12 train 0.1107 val 0.1514 (18s) best 0.1363
    ep 14 train 0.1060 val 0.1536 (20s) best 0.1363
  -> seed 7 test RMSE 65.66

[no-graph (no wind)] seed 123
    ep  4 train 0.1353 val 0.1317 (19s) best 0.1317
    ep  8 train 0.1222 val 0.1385 (19s) best 0.1317
    ep 12 train 0.1106 val 0.1488 (19s) best 0.1317
    ep 16 train 0.1019 val 0.1492 (18s) best 0.1317
  -> seed 123 test RMSE 67.32

[no-graph (no wind)] seed 2024
  

In [6]:
# CELL 6 - results table + save
# HOW TO CHECK: prints a table. Compare the three "ens RMSE" values; "seed std" is the noise level.
ar = ctx['from_log'](actual)
fut_spd = np.array([ctx['era5_spd'][s + INPUT_WINDOW:s + INPUT_WINDOW + HORIZON].mean() for s in ctx['te_starts']])
w33, w66 = np.percentile(fut_spd, [33, 66])
wreg = {'Calm': fut_spd <= w33, 'Breezy': (fut_spd > w33) & (fut_spd <= w66), 'Windy': fut_spd > w66}
sw = ar.max(1) - ar.min(1)
q33, q66 = np.percentile(sw, [33, 66])
vreg = {'Stable': sw <= q33, 'Moderate': (sw > q33) & (sw <= q66), 'Volatile': sw > q66}

res = {}
for label, ps in preds.items():
    per_seed = [metrics(ctx, p, actual)['rmse'] for p in ps]
    ens = metrics(ctx, np.mean(ps, axis=0), actual)
    pr = ctx['from_log'](np.mean(ps, axis=0))
    res[label] = dict(
        ens=ens, per_seed_rmse=per_seed,
        mean_seed_rmse=float(np.mean(per_seed)), std_seed_rmse=float(np.std(per_seed)),
        by_forecast_wind={k: float(np.sqrt(((ar[m] - pr[m]) ** 2).mean())) for k, m in wreg.items()},
        by_volatility={k: float(np.sqrt(((ar.transpose(0, 2, 1)[m] - pr.transpose(0, 2, 1)[m]) ** 2).mean()))
                       for k, m in vreg.items()})

json.dump({'note': 'Matched control for v3b: identical 23 features (no wind inputs, calendar-only decoder). '
                   'Only the graph differs. Seeds 42/7/123/2024/999.', 'results': res},
          open(os.path.join(RESULTS_DIR, 'v3b_controls.json'), 'w'), indent=2)

print('=== v3b matched controls (test, 5-seed ensembles) ===')
print(f"{'model':32}{'ens RMSE':>10}{'MAE':>8}{'R2':>8}{'seed mean':>11}{'seed std':>10}")
for label, r in res.items():
    print(f"{label:32}{r['ens']['rmse']:>10.2f}{r['ens']['mae']:>8.2f}{r['ens']['r2']:>8.3f}"
          f"{r['mean_seed_rmse']:>11.2f}{r['std_seed_rmse']:>10.2f}")
print('\nby forecast wind (Calm / Breezy / Windy):')
for label, r in res.items():
    print(f"  {label:32}" + ''.join(f'{r["by_forecast_wind"][k]:>9.2f}' for k in wreg))
print('by volatility (Stable / Moderate / Volatile):')
for label, r in res.items():
    print(f"  {label:32}" + ''.join(f'{r["by_volatility"][k]:>9.2f}' for k in vreg))
print('\nper-seed test RMSE:')
for label, r in res.items():
    print(f"  {label:32}", [round(x, 2) for x in r['per_seed_rmse']])
print('\nsaved v3b_controls.json')


=== v3b matched controls (test, 5-seed ensembles) ===
model                             ens RMSE     MAE      R2  seed mean  seed std
v3b dynamic (35km advection)         62.80   40.48   0.611      65.12      1.09
no-graph (no wind)                   63.21   40.31   0.606      65.39      1.53
static-GCN (no wind)                 62.72   40.74   0.612      65.14      1.35

by forecast wind (Calm / Breezy / Windy):
  v3b dynamic (35km advection)        85.69    51.20    43.92
  no-graph (no wind)                  86.96    51.52    42.84
  static-GCN (no wind)                84.80    50.90    45.55
by volatility (Stable / Moderate / Volatile):
  v3b dynamic (35km advection)        28.17    44.12    94.55
  no-graph (no wind)                  27.63    43.33    95.85
  static-GCN (no wind)                28.87    44.33    94.10

per-seed test RMSE:
  v3b dynamic (35km advection)     [65.7, 65.63, 63.83, 63.87, 66.57]
  no-graph (no wind)               [63.33, 65.66, 67.32, 63.98, 66.68]
  s